In [1]:
import sys
import numpy as np

# sys.path.append('../../../src/')
from Rain.Rain import Rain
# sys.path.pop()

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-06 22:21:19.259333: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-06 22:21:20.510089: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
config = {
  "mode": {
      "type": "cloud",
      "params": {
          "num_of_workers": 1,
          "subscription_id": "a7ef3688-af58-4835-953c-e51f219fbd0f", # Mostafa's ID
          # "subscription_id":'6e14c264-a7fc-4db4-a23a-d972c21a2d99', # Menna's ID
          # "subscription_id": '82305756-d4a0-442d-8e73-625e1ced2113', # Nada's ID
          "location": 'eastus'
      }
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 2,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 16,
  }
}

In [3]:
def get_train_data():
    return np.load("../../../data/breast_cancer/train_data.npy"), np.load(
        "../../../data/breast_cancer/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/breast_cancer/test_data.npy"), np.load(
        "../../../data/breast_cancer/test_labels.npy"
    )



In [4]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 30
    num_labels = 2
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [5]:
X_train, y_train = get_train_data()
X_train = np.reshape(X_train, [-1, 30])
y_train = to_categorical(y_train)

In [6]:
model = create_model()
rain = Rain(config, model)

2023-07-06 22:21:22,416 [DEBUG] [Rain] Rain is initialized
2023-07-06 22:21:22,417 [DEBUG] [Provisioner] Creating coordinator
2023-07-06 22:21:22,418 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-06 22:21:22,419 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-06 22:21:22,432 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/prov/
2023-07-06 22:21:22,438 [DEBUG] [CloudProvisioner] Provisioner is initialized
2023-07-06 22:21:22,447 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


2023-07-06 22:21:22,456 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-06 22:21:22,463 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [7]:
# model = rain.train(X_train, y_train, strategy='async')

In [8]:
# X_test, y_test = get_test_data()
# X_test = np.reshape(X_test, [-1, 30])
# y_test = to_categorical(y_test)
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [9]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-06 22:21:22,550 [INFO] [Provisioner] provisioner is serving
2023-07-06 22:21:22,553 [DEBUG] [Provisioner] Starting coordinator
2023-07-06 22:21:22,556 [INFO] [Coordinator] coordinator is serving
2023-07-06 22:21:22,558 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-06 22:21:22,578 [DEBUG] [Provisioner] Received 'NumOfWorkers: 1
' from the coordinator to define the number of workers
2023-07-06 22:21:22,583 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-06 22:21:22,588 [DEBUG] [CloudProvisioner] Setup the networking for the workers
2023-07-06 22:21:30,000 [DEBUG] [CloudProvisioner] Created resource group: Rain-resourcegroup
2023-07-06 22:21:40,051 [DEBUG] [CloudProvisioner] Created virtual network: Rain-vnet
2023-07-06 22:21:44,664 [DEBUG] [CloudProvisioner] Created network security group: Rain-nic-nsg
2023-07-06 22:21:44,668 [DEBUG] [CloudProvisioner] Network setup completed
2023-07-06 22:21:44,672 [DEBU

In [13]:
X_test, y_test = get_test_data()
X_test = np.reshape(X_test, [-1, 30])
y_test = to_categorical(y_test)
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

8/8 [==============================] - 0s 3ms/step - loss: 0.2842 - accuracy: 0.9386

Test accuracy: 93.9%
